# AASIST vs SSL-AASIST evaluation (folder-based, no zips)

Reads plain folders from your Google Drive, downloads the big stuff (fairseq, XLS-R) inside Colab,
and runs your `eval.py` for **both models on the same real 400-file subset**.

**Drive folder must contain:** `data/LA/...` (real subset), `SSL_Anti-spoofing-main/` (with `model.py`),
and `eval.py`, `spoof.py`, `aasist/` (anywhere inside that folder tree).

**Rules:** this notebook never generates fake audio, and stops if the AASIST baseline is not 2.39%.
Runtime: Runtime > Change runtime type > **T4 GPU**.

In [1]:
!nvidia-smi | head -15

Tue Nov  3 18:10:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Copy your project from Drive

In [3]:
import os, shutil, glob
from google.colab import drive
drive.mount("/content/drive")

DRIVE_FOLDER = "/content/drive/MyDrive/project"

ROOT = "/content/drive/MyDrive"
if not DRIVE_FOLDER:
    cands = []
    for dp, dn, fn in os.walk(ROOT):
        if dp[len(ROOT):].count(os.sep) > 2:
            dn[:] = []
            continue
        if "data" in dn and "SSL_Anti-spoofing-main" in dn:
            cands.append(dp)
    print("candidates:", cands)
    assert cands, "Could not find a Drive folder holding both 'data' and 'SSL_Anti-spoofing-main'. Set DRIVE_FOLDER."
    DRIVE_FOLDER = cands[0]
print("using:", DRIVE_FOLDER)

PROJ_ROOT = "/content/proj"
shutil.rmtree(PROJ_ROOT, ignore_errors=True)
shutil.copytree(DRIVE_FOLDER, PROJ_ROOT,
                ignore=shutil.ignore_patterns("fairseq-*", "*.zip", ".git", "__pycache__"))
print("copied. top level:", os.listdir(PROJ_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
using: /content/drive/MyDrive/project
copied. top level: ['SSL_Anti-spoofing-main', 'data']


## 2. Locate everything

In [4]:
def find_file(name, root, skip=("fairseq",), must_contain=None):
    for p in sorted(glob.glob(f"{root}/**/{name}", recursive=True), key=len):
        if any(s in p for s in skip):
            continue
        if must_contain and must_contain not in open(p, errors="ignore").read():
            continue
        return p
    return None

eval_py  = find_file("eval.py", PROJ_ROOT)
spoof_py = find_file("spoof.py", PROJ_ROOT)
model_py = find_file("model.py", PROJ_ROOT, must_contain="class SSLModel")
aasist_dirs = [d for d in glob.glob(f"{PROJ_ROOT}/**/aasist", recursive=True) if os.path.isdir(d)]
la_eval  = glob.glob(f"{PROJ_ROOT}/**/ASVspoof2019_LA_eval", recursive=True)

problems = []
if not eval_py:       problems.append("eval.py not found in your Drive folder")
if not spoof_py:      problems.append("spoof.py not found in your Drive folder")
if not aasist_dirs:   problems.append("aasist/ folder (code + AASIST.pth) not found")
if not model_py:      problems.append("SSL-AASIST model.py (with class SSLModel) not found")
if not la_eval:       problems.append("data/LA/ASVspoof2019_LA_eval not found")
assert not problems, "Upload these to the same Drive folder and rerun:\n- " + "\n- ".join(problems)

PROJ    = os.path.dirname(eval_py)
SSL_DIR = os.path.dirname(model_py)
LA_ROOT = os.path.dirname(la_eval[0])
print("PROJ   :", PROJ)
print("SSL_DIR:", SSL_DIR)
print("LA_ROOT:", LA_ROOT)
assert os.path.isdir(os.path.join(LA_ROOT, "ASVspoof2019_LA_eval", "flac")), "flac folder missing"
assert os.path.isfile(os.path.join(LA_ROOT, "ASVspoof2019_LA_cm_protocols", "ASVspoof2019.LA.cm.eval.trl.txt")), "protocol file missing"
print("AASIST weights:", [os.path.basename(p) for p in glob.glob(f"{PROJ_ROOT}/**/*.pth", recursive=True) if "fairseq" not in p])

PROJ   : /content/proj/SSL_Anti-spoofing-main
SSL_DIR: /content/proj/SSL_Anti-spoofing-main
LA_ROOT: /content/proj/data/LA
AASIST weights: ['AASIST-L.pth', 'AASIST.pth']


## 3. Guard: make sure the audio is real (not silence or placeholders)

In [5]:
import numpy as np, soundfile as sf

flacs = sorted(glob.glob(os.path.join(LA_ROOT, "ASVspoof2019_LA_eval", "flac", "*.flac")))
sizes = np.array([os.path.getsize(f) for f in flacs])
print("files:", len(flacs), "| size KB min/median/max:",
      int(sizes.min()/1e3), int(np.median(sizes)/1e3), int(sizes.max()/1e3))
rms = [float(np.sqrt(np.mean(np.square(sf.read(f)[0])))) for f in flacs[:20]]
print("RMS first clips:", [round(r, 4) for r in rms[:5]])

assert len(flacs) >= 100, "Too few audio files."
assert np.median(sizes) > 20_000, "Files are tiny: probably placeholders."
assert len(set(sizes.tolist())) > 50, "Many identical sizes: probably mock data."
assert min(rms) > 1e-5, "Silent audio found: probably mock data."
print("PASS: audio looks real")

files: 400 | size KB min/median/max: 16 51 205
RMS first clips: [0.0685, 0.0897, 0.1296, 0.2123, 0.1303]
PASS: audio looks real


## 4. Python 3.9 environment + fairseq (fetched from GitHub)

In [6]:
!pip -q install uv
!uv venv --seed --python 3.9 /content/venv39
PY = "/content/venv39/bin/python"
!{PY} --version
!{PY} -m pip install -q "pip<24.1" "setuptools<66" wheel
!{PY} -m pip install -q "numpy==1.23.5" cython
!{PY} -m pip install -q torch==2.0.1 torchaudio==2.0.2
!{PY} -m pip install -q librosa==0.10.1 soundfile scikit-learn tqdm

Using CPython 3.9.25
Creating virtual environment with seed packages at: venv39
 + packaging==26.3
 + pip==26.0.1
 + setuptools==82.0.1
 + wheel==0.48.0
Activate with: source venv39/bin/activate
Python 3.9.25
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.9/619.9 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.1/557.1 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 7.4 MB/s eta 0:00:00


In [8]:
COMMIT = "a54021305d6b3c4c5959ac9395135f63202db8f1"   # same fairseq commit the SSL repo bundles
!cd /content && rm -rf fairseq-{COMMIT} fq.zip && wget -q -O fq.zip https://github.com/facebookresearch/fairseq/archive/{COMMIT}.zip && unzip -q fq.zip
FAIRSEQ = f"/content/fairseq-{COMMIT}"
!cd {FAIRSEQ} && {PY} -m pip install -q --no-build-isolation --editable ./
!{PY} -c "import fairseq, torch; print('fairseq', fairseq.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

  Checking if build backend supports build_editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 10.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.1/390.1 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 791.3/791.3 kB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.8/750.8 kB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 120.3 MB/s eta 0:00:00
  Building editable for fairseq (pyproject.toml) ... done
DEPRECATION: omegaconf 2.0.6 has a non-standard dependency s

## 5. Checkpoints
`xlsr2_300m.pt` downloads automatically. For the **SSL-AASIST weights** (`LA_model.pth`) set ONE of the options below
(the link is in `SSL_Anti-spoofing-main/README.md`).

In [30]:
WEIGHTS_URL = "https://drive.google.com/drive/folders/1c4ywztEVlYVijfwbGLl9OEa1SNtFKppB?usp=sharing"
WEIGHTS_DRIVE_PATH = "/content/drive/MyDrive/project/SSL_Anti-spoofing-main/LA_model.pth"

XLSR = os.path.join(SSL_DIR, "xlsr2_300m.pt")
# Clean up existing file if it's incomplete to ensure a fresh download
if os.path.exists(XLSR) and os.path.getsize(XLSR) <= 1e9:
    print("Incomplete XLS-R weight file detected. Removing to redownload...")
    os.remove(XLSR)

!test -f {XLSR} || wget --show-progress -O {XLSR} https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr2_300m.pt
assert os.path.exists(XLSR) and os.path.getsize(XLSR) > 1e9, "xlsr2_300m.pt looks incomplete"

# Let's inspect all .pth files in PROJ_ROOT to help locate the weights
all_pth = glob.glob(os.path.join(PROJ_ROOT, "**/*.pth"), recursive=True)
print("Found the following .pth files in your project directory:")
for p in all_pth:
    if os.path.exists(p) and "fairseq" not in p:
        print(f"- {p} ({round(os.path.getsize(p) / 1e6, 2)} MB)")

LA_W = os.path.join(SSL_DIR, "LA_model.pth")
if not os.path.exists(LA_W) or os.path.getsize(LA_W) <= 1e7:
    if os.path.exists(LA_W): os.remove(LA_W)

    # 1. First search if you uploaded it locally/copied it
    found_paths = [p for p in all_pth if os.path.exists(p) and "fairseq" not in p and os.path.getsize(p) > 1e7]
    if found_paths:
        print(f"Copying found weights {found_paths[0]} to {LA_W}...")
        shutil.copy(found_paths[0], LA_W)
    # 2. Check if it's in your Google Drive path
    elif WEIGHTS_DRIVE_PATH and os.path.exists(WEIGHTS_DRIVE_PATH) and os.path.getsize(WEIGHTS_DRIVE_PATH) > 1e7:
        print(f"Copying weights from Drive: {WEIGHTS_DRIVE_PATH}...")
        shutil.copy(WEIGHTS_DRIVE_PATH, LA_W)
    # 3. Fallback: Download the official folder from Drive containing the weights
    else:
        print("Downloading LA_model.pth via gdown folder extraction...")
        folder_id = "1c4ywztEVlYVijfwbGLl9OEa1SNtFKppB"
        out_dir = "/tmp/gdrive_folder"
        os.makedirs(out_dir, exist_ok=True)

        # Use gdown --folder to download the contents of the shared folder
        !gdown --folder {folder_id} -O {out_dir} --remaining-ok

        # Check if LA_model.pth was downloaded
        downloaded_weights = glob.glob(os.path.join(out_dir, "**/LA_model.pth"), recursive=True)
        if downloaded_weights:
            print(f"Weights found at {downloaded_weights[0]}. Copying to project...")
            shutil.copy(downloaded_weights[0], LA_W)
        else:
            # Try matching any large .pth file inside downloaded folders
            any_weights = [p for p in glob.glob(os.path.join(out_dir, "**/*.pth"), recursive=True) if os.path.getsize(p) > 1e7]
            if any_weights:
                print(f"Found large weights at {any_weights[0]}. Copying to project...")
                shutil.copy(any_weights[0], LA_W)

assert os.path.exists(LA_W) and os.path.getsize(LA_W) > 1e7, \
    "SSL-AASIST weights missing. Please manually download LA_model.pth and upload it to your Google Drive project folder."
print("weights OK:", round(os.path.getsize(LA_W) / 1e6), "MB")

# make model.py find xlsr2_300m.pt next to itself
import re
mp = os.path.join(SSL_DIR, "model.py")
src = open(mp).read()
if "os.path.abspath(__file__)" not in src:
    src = re.sub(r"cp_path\s*=\s*['\"][^'\"]*['\"]",
                 "cp_path = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'xlsr2_300m.pt')", src)
    if "import os" not in src:
        src = "import os\n" + src
    open(mp, "w").write(src)
print([l for l in open(mp).read().splitlines() if "cp_path" in l][:2])

Found the following .pth files in your project directory:
- /content/proj/SSL_Anti-spoofing-main/aasist/models/weights/AASIST-L.pth (0.43 MB)
- /content/proj/SSL_Anti-spoofing-main/aasist/models/weights/AASIST.pth (1.28 MB)
Retrieving folder contents
Processing file 1JHBClArVdM-Cr1b8In1iakTV_TvC3HvG Best_LA_model_for_DF.pth
Processing file 11vFBNKzYUtWqdK358_JEygawFzmmaZjO LA_model.pth
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1JHBClArVdM-Cr1b8In1iakTV_TvC3HvG
From (redirected): https://drive.google.com/uc?id=1JHBClArVdM-Cr1b8In1iakTV_TvC3HvG&confirm=t&uuid=15266fc7-1a7b-4225-abc4-13bca5a26bba
To: /tmp/gdrive_folder/Best_LA_model_for_DF.pth
100% 1.27G/1.27G [00:25<00:00, 49.5MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=11vFBNKzYUtWqdK358_JEygawFzmmaZjO
From (redirected): https://drive.google.com/uc?id=11vFBNKzYUtWqdK358_JEygawFzmmaZjO&con

In [22]:
import re
import os

# Scan text files in the project root for URLs containing model weights
search_patterns = [
    re.compile(r'https://drive\.google\.com/[^\s"\)\]]+'),
    re.compile(r'https://[\w\d./_-]+/LA_model\.pth'),
    re.compile(r'id=[\w\d_-]+')
]

print("Searching project text files for Google Drive / download links...")
found_any = False

for root, _, files in os.walk(PROJ_ROOT):
    for file in files:
        if file.endswith((".md", ".txt", ".sh", ".py")) and "venv" not in root and "fairseq" not in root:
            filepath = os.path.join(root, file)
            try:
                content = open(filepath, errors="ignore").read()
                if "LA_model" in content or "model" in content or "drive" in content:
                    lines = content.splitlines()
                    for idx, line in enumerate(lines):
                        for pattern in search_patterns:
                            matches = pattern.findall(line)
                            if matches:
                                rel_path = os.path.relpath(filepath, PROJ_ROOT)
                                print(f"\n[In {rel_path} (line {idx+1})]:")
                                print(f"  {line.strip()}")
                                found_any = True
            except Exception:
                pass

if not found_any:
    print("No links found in the local repository files. Please check the original repository's README on GitHub or your sources for the LA_model.pth download link.")

Searching project text files for Google Drive / download links...

[In SSL_Anti-spoofing-main/README.md (line 56)]:
  Pre-trained SSL antispoofing models are available for LA and DF [here](https://drive.google.com/drive/folders/1c4ywztEVlYVijfwbGLl9OEa1SNtFKppB?usp=sharing)


## 6. Write `spoof_ssl.py` and make sure `eval.py` has `--model`

In [31]:
SPOOF_SSL = '''
\"\"\"SSL-AASIST detector with the same interface as spoof.py.\"\"\"
import os, sys, glob
from argparse import Namespace
import numpy as np
import torch
import librosa

HERE = os.path.dirname(os.path.abspath(__file__))

def _find_ssl_dir():
    cands = [HERE] + [os.path.join(HERE, d) for d in os.listdir(HERE)]
    for root, _, files in os.walk(HERE):
        if "fairseq" not in root:
            cands.append(root)
    for c in cands:
        p = os.path.join(c, "model.py")
        if os.path.isfile(p) and "class SSLModel" in open(p, errors="ignore").read():
            return c
    raise FileNotFoundError("SSL-AASIST model.py not found under " + HERE)

SSL_DIR = _find_ssl_dir()
sys.path.insert(0, SSL_DIR)
from model import Model  # noqa: E402

SR = 16000
MAX_LEN = 64600

def _weights():
    p = os.path.join(SSL_DIR, "LA_model.pth")
    if os.path.exists(p):
        return p
    c = sorted(glob.glob(os.path.join(SSL_DIR, "*.pth")))
    if not c:
        raise FileNotFoundError("No .pth weights in " + SSL_DIR)
    return c[0]

def load_model(device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = Model(Namespace(), device).to(device)
    state = torch.load(_weights(), map_location=device)
    state = {(k[7:] if k.startswith("module.") else k): v for k, v in state.items()}
    model.load_state_dict(state)
    model.eval()
    return model, device

def _pad(x, n=MAX_LEN):
    if len(x) >= n:
        return x[:n]
    return np.tile(x, int(np.ceil(n / len(x))))[:n]

def _chunks(wav):
    if len(wav) <= MAX_LEN:
        return [_pad(wav)]
    out = [wav[i:i + MAX_LEN] for i in range(0, len(wav) - MAX_LEN + 1, MAX_LEN)]
    rest = len(wav) % MAX_LEN
    if rest > SR:
        out.append(_pad(wav[-rest:]))
    return out

@torch.no_grad()
def spoof_scores(path, model, device):
    wav, _ = librosa.load(path, sr=SR, mono=True)
    if len(wav) == 0:
        raise ValueError("Empty audio: " + path)
    batch = torch.tensor(np.stack(_chunks(wav)), dtype=torch.float32).to(device)
    out = model(batch)  # (n_chunks, 2); index 1 = bonafide
    return out[:, 1].mean().item(), torch.softmax(out, dim=1)[:, 0].mean().item()

if __name__ == "__main__":
    m, d = load_model()
    s, p = spoof_scores(sys.argv[1], m, d)
    print(f"bonafide score: {s:.3f} | spoof probability: {p:.2%}")
'''
open(os.path.join(PROJ, "spoof_ssl.py"), "w").write(SPOOF_SSL.lstrip())

ep = os.path.join(PROJ, "eval.py")
e = open(ep).read()
if '"--model"' not in e:
    e = e.replace("from spoof import load_model, spoof_scores\n", "")
    e = e.replace('ap.add_argument("--seed", type=int, default=42)',
                  'ap.add_argument("--seed", type=int, default=42)\n'
                  '    ap.add_argument("--model", choices=["aasist", "ssl"], default="aasist")')
    e = e.replace("args = ap.parse_args()",
                  "args = ap.parse_args()\n"
                  "    if args.model == 'ssl':\n"
                  "        from spoof_ssl import load_model, spoof_scores\n"
                  "    else:\n"
                  "        from spoof import load_model, spoof_scores")
    open(ep, "w").write(e)
    print("patched eval.py")
else:
    print("eval.py already has --model")
!cd {PROJ} && {PY} eval.py -h | head -12

eval.py already has --model
usage: eval.py [-h] --root ROOT [--n N] [--seed SEED] [--model {aasist,ssl}]

optional arguments:
  -h, --help            show this help message and exit
  --root ROOT           path to the LA folder
  --n N                 subset size
  --seed SEED
  --model {aasist,ssl}


## 7. Baseline check: AASIST must reproduce your laptop result (2.39%)

In [32]:
!cd {PROJ} && {PY} eval.py --root {LA_ROOT} --n 400 --model aasist 2>&1 | tee /content/aasist_results.txt

100%|██████████| 400/400 [00:20<00:00, 19.31it/s]

Samples: 400 (bonafide=39, spoof=361)
Overall EER: 2.39%
  A07: EER 0.00%
  A08: EER 0.00%
  A09: EER 0.00%
  A10: EER 2.95%
  A11: EER 0.00%
  A12: EER 3.01%
  A13: EER 0.00%
  A14: EER 0.00%
  A15: EER 2.75%
  A16: EER 0.00%
  A17: EER 3.13%
  A18: EER 8.58%
  A19: EER 0.00%


In [33]:
import re
t = open("/content/aasist_results.txt").read()
eer = float(re.search(r"Overall EER:\s*([0-9.]+)%", t).group(1))
nb_, ns_ = map(int, re.search(r"bonafide=(\d+), spoof=(\d+)", t).groups())
print("AASIST EER:", eer, "| bonafide/spoof:", nb_, ns_)
assert (nb_, ns_) == (39, 361), "Different files than your laptop run: check data/LA"
assert abs(eer - 2.39) < 0.3, "Baseline does not match 2.39%: data or environment problem. Do NOT continue."
print("Baseline reproduced. Safe to run SSL-AASIST.")

AASIST EER: 2.39 | bonafide/spoof: 39 361
Baseline reproduced. Safe to run SSL-AASIST.


## 8. SSL-AASIST sanity check on one real and one fake clip

In [34]:
proto = os.path.join(LA_ROOT, "ASVspoof2019_LA_cm_protocols", "ASVspoof2019.LA.cm.eval.trl.txt")
flac_dir = os.path.join(LA_ROOT, "ASVspoof2019_LA_eval", "flac")
real_f = fake_f = None
for line in open(proto):
    p = line.split()
    if len(p) < 5: continue
    f = os.path.join(flac_dir, p[1] + ".flac")
    if not os.path.exists(f): continue
    if p[4] == "bonafide" and real_f is None: real_f = f
    if p[4] == "spoof" and fake_f is None: fake_f = f
    if real_f and fake_f: break
print("REAL:", real_f)
!cd {PROJ} && {PY} spoof_ssl.py {real_f}
print("FAKE:", fake_f)
!cd {PROJ} && {PY} spoof_ssl.py {fake_f}

REAL: /content/proj/data/LA/ASVspoof2019_LA_eval/flac/LA_E_9156158.flac
2026-11-03 18:52:38 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
bonafide score: 5.586 | spoof probability: 0.00%
FAKE: /content/proj/data/LA/ASVspoof2019_LA_eval/flac/LA_E_2184116.flac
2026-11-03 18:53:20 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
bonafide score: -4.770 | spoof probability: 100.00%


Expected: the **real** clip has a *low* spoof probability, the **fake** clip a *high* one. If reversed, tell me before continuing.

## 9. SSL-AASIST evaluation

In [35]:
!cd {PROJ} && {PY} eval.py --root {LA_ROOT} --n 20 --model ssl 2>&1 | tail -8

2026-11-03 18:54:08 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
100%|██████████| 20/20 [00:07<00:00,  2.59it/s]

Samples: 20 (bonafide=3, spoof=17)
Overall EER: 0.00%


In [36]:
!cd {PROJ} && {PY} eval.py --root {LA_ROOT} --n 400 --model ssl 2>&1 | tee /content/ssl_results.txt

2026-11-03 18:55:01 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
100%|██████████| 400/400 [00:32<00:00, 12.33it/s]

Samples: 400 (bonafide=39, spoof=361)
Overall EER: 0.00%
  A07: EER 0.00%
  A08: EER 0.00%
  A09: EER 0.00%
  A10: EER 0.00%
  A11: EER 0.00%
  A12: EER 0.00%
  A13: EER 0.00%
  A14: EER 0.00%
  A15: EER 0.00%
  A16: EER 0.00%
  A17: EER 0.00%
  A18: EER 0.00%
  A19: EER 0.00%


## 10. Side-by-side table

In [37]:
def parse(path):
    t = open(path).read()
    d = {"Overall": float(re.search(r"Overall EER:\s*([0-9.]+)%", t).group(1))}
    for a, v in re.findall(r"(A\d\d): EER ([0-9.]+)%", t):
        d[a] = float(v)
    return d

A, S = parse("/content/aasist_results.txt"), parse("/content/ssl_results.txt")
print(f"{'':10s}{'AASIST':>10s}{'SSL-AASIST':>14s}")
for k in A:
    print(f"{k:10s}{A[k]:>9.2f}%{S.get(k, float('nan')):>13.2f}%")

              AASIST    SSL-AASIST
Overall        2.39%         0.00%
A07            0.00%         0.00%
A08            0.00%         0.00%
A09            0.00%         0.00%
A10            2.95%         0.00%
A11            0.00%         0.00%
A12            3.01%         0.00%
A13            0.00%         0.00%
A14            0.00%         0.00%
A15            2.75%         0.00%
A16            0.00%         0.00%
A17            3.13%         0.00%
A18            8.58%         0.00%
A19            0.00%         0.00%


In [38]:
from google.colab import files
files.download("/content/aasist_results.txt")
files.download("/content/ssl_results.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>